# 02 - Feature Pipeline

Builds the model-ready feature matrix from the raw store: returns, rolling volatility, momentum, and the aligned macro block, followed by the outlier screen and the date-based train/validation/test split. Every parameter comes from `config/config.yaml` so the split dates are identical across all experiments. Outputs are written once and read by notebooks 03-05.

In [ ]:
import yaml
import numpy as np
import pandas as pd

with open("../config/config.yaml") as f:
    CFG = yaml.safe_load(f)

prices = pd.read_csv("../data/raw/prices.csv", index_col=0, parse_dates=True)
macro = pd.read_csv("../data/raw/macro.csv", index_col=0, parse_dates=True)

# Calendar alignment across tickers
prices = prices.sort_index().ffill(limit=CFG["cleaning"]["ffill_limit"]).dropna(how="all")
print("Aligned price matrix:", prices.shape)

## 1. Core features

Daily simple returns feed the LSTM directly; annualized rolling volatility and multi-horizon momentum capture the persistence the networks are expected to exploit (Problem Statement, Section 3.1).

In [ ]:
rets = prices.pct_change().dropna()
vol = rets.rolling(CFG["features"]["vol_window"]).std() * np.sqrt(252)
mom = rets.rolling(CFG["features"]["mom_window"]).sum()

# Macro block: forward-fill limited to the configured policy, then reindex
macro = macro.sort_index().ffill(limit=CFG["cleaning"]["ffill_limit"]).reindex(rets.index)
features = pd.concat({"ret": rets, "vol": vol, "mom": mom}, axis=1)
print("Feature blocks:", features.columns.levels[0].tolist())
print("Macro series aligned:", macro.columns.tolist())

## 2. Outlier screen

Winsorization at the configured quantile, with the rule fixed before the test block is touched (pre-commitment against data snooping).

In [ ]:
q_lo, q_hi = CFG["cleaning"]["winsor"]
screened = features.clip(lower=features.quantile(q_lo), upper=features.quantile(q_hi))
print("Values clipped:", int((features != screened).sum().sum()))

## 3. Date-based splits

Train/validation/test boundaries come from the config, not from data. The test block remains untouched until the final walk-forward run.

In [ ]:
splits = CFG["splits"]
train = screened.loc[splits["train_start"]:splits["train_end"]]
valid = screened.loc[splits["valid_start"]:splits["valid_end"]]
test = screened.loc[splits["test_start"]:]
for name, frame in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name}: {frame.index.min().date()} -> {frame.index.max().date()}  rows={len(frame)}")
assert not test.empty, "Test block empty - check split dates in config."

## 4. Persist

Parquet is used for the feature matrix to preserve dtypes. The cleaning log below is the artifact the report cites.

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
screened.to_parquet("../data/processed/features.parquet")
macro.to_csv("../data/processed/macro_aligned.csv")
log = pd.DataFrame({
    "step": ["ffill_limit", "winsor_low", "winsor_high", "rows", "tickers"],
    "value": [CFG["cleaning"]["ffill_limit"], q_lo, q_hi, len(screened), prices.shape[1]],
})
log.to_csv("../data/processed/cleaning_log.csv", index=False)
print(log)